In [73]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(context='notebook', style='darkgrid', font='sans-serif')

In [74]:
URL = "https://raw.githubusercontent.com/plotly/datasets/master/winequality-red.csv"
df = pd.read_csv(URL)

In [75]:
df.shape

(1599, 12)

In [76]:
df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [77]:
df = df.drop_duplicates()

In [78]:
df.shape

(1359, 12)

In [79]:
df.describe()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
count,1359.000000,1359.000000,1359.000000,1359.000000,1359.000000,1359.000000,1359.000000,1359.000000,1359.000000,1359.000000,1359.000000,1359.000000
mean,8.310596,0.529478,0.272333,2.523400,0.088124,15.893304,46.825975,0.996709,3.309787,0.658705,10.432315,5.623252
std,1.736990,0.183031,0.195537,1.352314,0.049377,10.447270,33.408946,0.001869,0.155036,0.170667,1.082065,0.823578
min,4.600000,0.120000,0.000000,0.900000,0.012000,1.000000,6.000000,0.990070,2.740000,0.330000,8.400000,3.000000
25%,7.100000,0.390000,0.090000,1.900000,0.070000,7.000000,22.000000,0.995600,3.210000,0.550000,9.500000,5.000000
50%,7.900000,0.520000,0.260000,2.200000,0.079000,14.000000,38.000000,0.996700,3.310000,0.620000,10.200000,6.000000
75%,9.200000,0.640000,0.430000,2.600000,0.091000,21.000000,63.000000,0.997820,3.400000,0.730000,11.100000,6.000000
max,15.900000,1.580000,1.000000,15.500000,0.611000,72.000000,289.000000,1.003690,4.010000,2.000000,14.900000,8.000000


In [80]:
df.columns

Index(['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
       'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density',
       'pH', 'sulphates', 'alcohol', 'quality'],
      dtype='str')

In [81]:
df['good'] = (df['quality'] >= 6).astype(int)

In [82]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['good', 'quality'])

y = df['good']

X_train, X_test, y_train, y_test = train_test_split(
	X, y, 
    test_size=0.2,       
    random_state=42,     
    stratify=y          
)

In [83]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

acc = [(t.predict(X_test.values) == y_test.values).mean() for t in rf.estimators_]
acc = np.array(acc)

print(f"Min -> {np.round(acc.min(), 4)}")
print(f"Mean -> {np.round(acc.mean(), 4)}")
print(f"Max -> {np.round(acc.max(), 4)}")
print(f"Test Accuracy:  -> {np.round(rf.score(X_test, y_test), 4)}")
print(f"Train Accuracy: -> {np.round(rf.score(X_train, y_train), 4)}")

Min -> 0.6103
Mean -> 0.6598
Max -> 0.7206
Test Accuracy:  -> 0.75
Train Accuracy: -> 1.0


In [84]:
from sklearn.metrics import classification_report

y_pred = rf.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.72      0.77      0.74       128
           1       0.78      0.74      0.76       144

    accuracy                           0.75       272
   macro avg       0.75      0.75      0.75       272
weighted avg       0.75      0.75      0.75       272



In [85]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train, y_train)


print(f"Train Accuracy: {dt.score(X_train, y_train):.4f}")
print(f"Test Accuracy:  {dt.score(X_test, y_test):.4f}\n")

Train Accuracy: 0.8031
Test Accuracy:  0.6765



In [86]:
from sklearn.model_selection import cross_val_score, KFold

cv = KFold(n_splits=5, shuffle=True, random_state=42)

print(f"Random Forest Mean Accuracy: {cross_val_score(rf, X, y, cv=cv).mean():.2f}")
print(f"Decision Tree Mean Accuracy: {cross_val_score(dt, X, y, cv=cv).mean():.2f}")

Random Forest Mean Accuracy: 0.76
Decision Tree Mean Accuracy: 0.71


In [87]:
from sklearn.metrics import confusion_matrix

models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42)
}

for name, m in models.items():
    m.fit(X_train, y_train)
    
    y_pred = m.predict(X_test)
    print(f"=== {name} ===")
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    
    cv_mean = cross_val_score(m, X, y, cv=5).mean()
    print(f"5-Fold CV Mean Accuracy: {cv_mean:.4f}\n")

=== Decision Tree ===
Confusion Matrix:
[[75 53]
 [49 95]]
5-Fold CV Mean Accuracy: 0.6475

=== Random Forest ===
Confusion Matrix:
[[ 98  30]
 [ 38 106]]
5-Fold CV Mean Accuracy: 0.7233



In [88]:
for k in [3, 5, 10, None]:
	m = RandomForestClassifier(max_depth=k, random_state=42)
	m.fit(X_train, y_train)
	print(f"{k} -> Train {round(m.score(X_train, y_train), 4)} -> Test {round(m.score(X_test, y_test), 4)}")

# train va test farqi eng kam bo'lgani bizga muhum hisoblanadi yani k=5 

3 -> Train 0.77 -> Test 0.7537
5 -> Train 0.8123 -> Test 0.7647
10 -> Train 0.9834 -> Test 0.7574
None -> Train 1.0 -> Test 0.75


In [89]:
for n in [1, 10, 100]:
    rf = RandomForestClassifier(n_estimators=n, random_state=42)
    rf.fit(X_train, y_train)
    test_acc = rf.score(X_test, y_test)
    train_acc = rf.score(X_train, y_train)
    print(f"n_estimators={n} -> Test Accuracy: {test_acc:.4f}")
    print(f"n_estimators={n} -> Train Accuracy: {test_acc:.4f}")
    print('-'*40)


n_estimators=1 -> Test Accuracy: 0.6985
n_estimators=1 -> Train Accuracy: 0.6985
----------------------------------------
n_estimators=10 -> Test Accuracy: 0.7537
n_estimators=10 -> Train Accuracy: 0.7537
----------------------------------------
n_estimators=100 -> Test Accuracy: 0.7500
n_estimators=100 -> Train Accuracy: 0.7500
----------------------------------------


In [90]:
for mf in ['sqrt', 0.5, None]:
    rf = RandomForestClassifier(max_features=mf, random_state=42)
    rf.fit(X_train, y_train)
    test_acc = rf.score(X_test, y_test)
    train_acc = rf.score(X_train, y_train)
    print(f"max_features={mf} -> Test Accuracy: {test_acc:.4f}")
    print(f"max_features={mf} -> Train Accuracy: {train_acc:.4f}")
    print('-'*40)

max_features=sqrt -> Test Accuracy: 0.7500
max_features=sqrt -> Train Accuracy: 1.0000
----------------------------------------
max_features=0.5 -> Test Accuracy: 0.7574
max_features=0.5 -> Train Accuracy: 1.0000
----------------------------------------
max_features=None -> Test Accuracy: 0.7463
max_features=None -> Train Accuracy: 1.0000
----------------------------------------


In [91]:
for md in [3, 5, 10, None]:
    rf = RandomForestClassifier(max_depth=md, random_state=42)
    rf.fit(X_train, y_train)
    test_acc = rf.score(X_test, y_test)
    train_acc = rf.score(X_train, y_train)
    print(f"max_depth={md} -> Test Accuracy: {test_acc:.4f}")
    print(f"max_depth={md} -> Train Accuracy: {train_acc:.4f}")
    print('-'*40)

max_depth=3 -> Test Accuracy: 0.7537
max_depth=3 -> Train Accuracy: 0.7700
----------------------------------------
max_depth=5 -> Test Accuracy: 0.7647
max_depth=5 -> Train Accuracy: 0.8123
----------------------------------------
max_depth=10 -> Test Accuracy: 0.7574
max_depth=10 -> Train Accuracy: 0.9834
----------------------------------------
max_depth=None -> Test Accuracy: 0.7500
max_depth=None -> Train Accuracy: 1.0000
----------------------------------------


In [93]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [10, 50, 100],
    'max_features': ['sqrt', 0.5, None],
    'max_depth': [3, 5, 10, None]
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,                  
    scoring='accuracy',
    n_jobs=-1           
)

grid_search.fit(X_train, y_train)

print(f"Best Params: {grid_search.best_params_}")
print(f"Best CV score: {grid_search.best_score_:.4f}")

best_rf = grid_search.best_estimator_
print(f"Latest Test accuracy: {best_rf.score(X_test, y_test):.4f}")

Best Params: {'max_depth': 10, 'max_features': 'sqrt', 'n_estimators': 50}
Best CV score: 0.7581
Latest Test accuracy: 0.7463


In [ ]:
# -------------------------------------------------------------------------------
# feature importance

# random forest
importance = pd.Series(rf.feature_importances_, index = X.columns).sort_values(ascending=False)
print(importance.head(3).round(4))

alcohol             0.2133
sulphates           0.1223
volatile acidity    0.1083
dtype: float64


In [ ]:
# decision tree
importance = pd.Series(dt.feature_importances_, index = X.columns).sort_values(ascending=False)
print(importance.head(3).round(4))

alcohol                 0.5416
sulphates               0.1427
total sulfur dioxide    0.0848
dtype: float64


In [ ]:
selected_features = ['alcohol', 'sulphates', 'total sulfur dioxide']

X_subset = df[selected_features]

X_train, X_test, y_train, y_test = train_test_split(
    X_subset, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

dt_subset = DecisionTreeClassifier(random_state=42)
dt_subset.fit(X_train, y_train)

print(f"Test Accuracy -> {round(dt_subset.score(X_test, y_test), 4)}")

# Test Accuracy:  -> 0.75 -> eski -> 11 ta feature bilan 
# Test Accuracy -> 0.7132 -> yangi -> 3 ta feature bilan

Test Accuracy -> 0.7132
